# Titanic Survival Classification

**Objective:** Build a machine learning model to predict whether a passenger survived or not based on features like age, gender, ticket class, and fare.

**Workflow:**
1. Load & explore the Titanic dataset
2. Clean data & handle missing values
3. Engineer features (Title, FamilySize, IsAlone, HasCabin)
4. Build preprocessing pipeline (scaling + encoding)
5. Train multiple classifiers (Logistic Regression, Random Forest, Gradient Boosting)
6. Evaluate & compare model performance
7. Analyze feature importance
8. Test with a sample passenger prediction

---
## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully!')

---
## 2. Load the Dataset

Download the Titanic dataset from a public GitHub source and cache it locally.

In [ ]:
# Configuration
DATA_DIR = os.path.join(os.getcwd(), 'data')
TITANIC_CSV = os.path.join(DATA_DIR, 'titanic.csv')
TITANIC_URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
RANDOM_STATE = 42
TEST_SIZE = 0.2

os.makedirs(DATA_DIR, exist_ok=True)

# Download if not cached
if not os.path.exists(TITANIC_CSV):
    print(f'Downloading Titanic dataset from {TITANIC_URL} ...')
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(TITANIC_URL, headers=headers, timeout=30)
    response.raise_for_status()
    with open(TITANIC_CSV, 'wb') as f:
        f.write(response.content)
    print(f'Saved to {TITANIC_CSV}')
else:
    print(f'Dataset already cached at {TITANIC_CSV}')

# Load into DataFrame
df = pd.read_csv(TITANIC_CSV)
print(f'\nDataset shape: {df.shape}')
df.head()

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
print(f'Shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nBasic statistics:')
df.describe()

In [ ]:
# Survival distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Survival count
sns.countplot(x='Survived', data=df, ax=axes[0], palette='viridis')
axes[0].set_title('Survival Count')
axes[0].set_xticklabels(['Not Survived', 'Survived'])

# Survival by Gender
sns.countplot(x='Sex', hue='Survived', data=df, ax=axes[1], palette='viridis')
axes[1].set_title('Survival by Gender')

# Survival by Class
sns.countplot(x='Pclass', hue='Survived', data=df, ax=axes[2], palette='viridis')
axes[2].set_title('Survival by Ticket Class')

plt.tight_layout()
plt.show()

In [ ]:
# Age and Fare distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Age'].dropna(), bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Age Distribution')

sns.histplot(df['Fare'], bins=30, kde=True, ax=axes[1], color='coral')
axes[1].set_title('Fare Distribution')

plt.tight_layout()
plt.show()

---
## 4. Data Cleaning & Feature Engineering

In [ ]:
# Title grouping map
TITLE_MAP = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Mme': 'Mrs', 'Ms': 'Miss', 'Lady': 'Rare',
    'the Countess': 'Rare', 'Capt': 'Rare', 'Sir': 'Rare',
    'Don': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare',
}

def extract_title(name):
    """Extracts and groups the title from a passenger name."""
    if pd.isna(name):
        return 'Mr'
    title = name.split(',')[1].split('.')[0].strip()
    return TITLE_MAP.get(title, 'Rare')

def engineer_features(df):
    """Creates derived features from the raw Titanic DataFrame."""
    df = df.copy()
    
    # Family Size & IsAlone
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    # Has Cabin indicator
    df['HasCabin'] = df['Cabin'].notna().astype(int)
    
    # Title extraction
    df['Title'] = df['Name'].apply(extract_title)
    
    # Drop columns not needed
    cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch']
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
    
    return df

# Apply feature engineering
X = df.drop(columns=['Survived'])
y = df['Survived']

X = engineer_features(X)
X['Pclass'] = X['Pclass'].astype(str)  # Treat Pclass as categorical

print(f'Engineered features shape: {X.shape}')
print(f'\nColumns: {list(X.columns)}')
X.head()

---
## 5. Preprocessing Pipeline

Build an sklearn `ColumnTransformer` that:
- **Numerical columns** (Age, Fare, FamilySize): Median imputation + Standard scaling
- **Categorical columns** (Sex, Embarked, Title, Pclass): Most-frequent imputation + One-hot encoding

In [ ]:
# Define column groups
NUMERICAL_COLS = ['Age', 'Fare', 'FamilySize']
CATEGORICAL_COLS = ['Sex', 'Embarked', 'Title', 'Pclass']

# Build pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, NUMERICAL_COLS),
        ('cat', cat_pipeline, CATEGORICAL_COLS),
    ]
)

print('Preprocessing pipeline built.')

---
## 6. Train-Test Split & Preprocessing

In [ ]:
# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Fit & transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Get feature names after encoding
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_features = list(cat_encoder.get_feature_names_out(CATEGORICAL_COLS))
feature_names = list(NUMERICAL_COLS) + cat_features

print(f'Training set   : {X_train_processed.shape}')
print(f'Test set       : {X_test_processed.shape}')
print(f'Total features : {len(feature_names)}')
print(f'\nFeature names: {feature_names}')

---
## 7. Model Training & Evaluation

Train three classifiers and compare their performance:
1. **Logistic Regression** - Linear baseline
2. **Random Forest** - Ensemble of decision trees
3. **Gradient Boosting** - Sequential ensemble learning

In [ ]:
# Define candidate models
candidate_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42
    ),
}

# Train & evaluate each model
results = {}

for name, model in candidate_models.items():
    print(f'\n{"=" * 55}')
    print(f'  Training: {name}')
    print(f'{"=" * 55}')
    
    model.fit(X_train_processed, y_train)
    
    y_pred = model.predict(X_test_processed)
    y_proba = model.predict_proba(X_test_processed)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)
    
    results[name] = {
        'model': model,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'roc_auc': auc,
        'confusion_matrix': cm,
        'y_pred': y_pred,
        'y_proba': y_proba,
    }
    
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print(f'  ROC-AUC   : {auc:.4f}')

# Identify best model
best_name = max(results, key=lambda k: results[k]['accuracy'])
print(f'\n{"=" * 55}')
print(f'  BEST MODEL: {best_name} (Accuracy: {results[best_name]["accuracy"]:.4f})')
print(f'{"=" * 55}')

### 7.1 Model Comparison Table

In [ ]:
# Comparison DataFrame
comparison_df = pd.DataFrame({
    name: {
        'Accuracy': r['accuracy'],
        'Precision': r['precision'],
        'Recall': r['recall'],
        'F1-Score': r['f1_score'],
        'ROC-AUC': r['roc_auc'],
    }
    for name, r in results.items()
}).T

comparison_df.style.highlight_max(axis=0, color='lightgreen').format('{:.4f}')

### 7.2 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, r) in zip(axes, results.items()):
    sns.heatmap(
        r['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
        xticklabels=['Not Survived', 'Survived'],
        yticklabels=['Not Survived', 'Survived'],
        ax=ax
    )
    ax.set_title(f'{name}\nAccuracy: {r["accuracy"]:.4f}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

### 7.3 ROC Curves

In [ ]:
plt.figure(figsize=(10, 7))

colors = ['#2563eb', '#16a34a', '#ea580c']
for (name, r), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {r["roc_auc"]:.4f})', color=color, lw=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 7.4 Classification Report (Best Model)

In [ ]:
best_model = results[best_name]['model']
y_pred_best = results[best_name]['y_pred']

print(f'Classification Report - {best_name}:\n')
print(classification_report(y_test, y_pred_best, target_names=['Not Survived', 'Survived']))

---
## 8. Feature Importance Analysis

In [ ]:
# Get feature importance for each model type
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (name, r) in zip(axes, results.items()):
    model = r['model']
    
    if name == 'Logistic Regression':
        importances = np.abs(model.coef_[0])
        label = '|Coefficient|'
    else:
        importances = model.feature_importances_
        label = 'Importance'
    
    # Sort by importance
    sorted_idx = np.argsort(importances)
    
    ax.barh(
        [feature_names[i] for i in sorted_idx],
        importances[sorted_idx],
        color='steelblue', alpha=0.8
    )
    ax.set_title(f'{name}\n{label}')
    ax.set_xlabel(label)

plt.tight_layout()
plt.show()

---
## 9. Sample Passenger Prediction

In [ ]:
# Define a test passenger
test_passenger = {
    'Pclass': 1,
    'Sex': 'female',
    'Age': 29.0,
    'SibSp': 0,
    'Parch': 0,
    'Fare': 100.0,
    'Embarked': 'S',
    'Name': 'Chaffee, Mrs. Herbert Shingler (Carrie Toogood)',
    'Cabin': 'E33',
}

# Preprocess the passenger
df_single = pd.DataFrame([test_passenger])
df_single = engineer_features(df_single)
df_single['Pclass'] = df_single['Pclass'].astype(str)

required = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked',
             'FamilySize', 'IsAlone', 'HasCabin', 'Title']
for col in required:
    if col not in df_single.columns:
        df_single[col] = np.nan
df_single = df_single[required]

processed = preprocessor.transform(df_single)

# Predict with all models
print('Sample Passenger Prediction')
print('=' * 50)
for name, r in results.items():
    model = r['model']
    pred = model.predict(processed)[0]
    proba = model.predict_proba(processed)[0][1]
    status = 'SURVIVED' if pred == 1 else 'DID NOT SURVIVE'
    print(f'  {name:25s} -> {status} (probability: {proba:.2%})')

print(f'\nPassenger details: {test_passenger}')

---
## Summary

- Loaded and cleaned the Titanic dataset (891 passengers)
- Engineered features: Title, FamilySize, IsAlone, HasCabin
- Built a robust preprocessing pipeline with imputation + encoding
- Trained and compared 3 classifiers:
  - **Logistic Regression** - Interpretable linear model
  - **Random Forest** - Robust ensemble method
  - **Gradient Boosting** - High-performance sequential ensemble
- Key finding: Gender (Sex) and Ticket Class (Pclass) are the strongest predictors of survival